In [0]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F

In [ ]:
staging_schema = "stocks.stage"

# Derive expected stage table names from the reference table
# (same transformation as stage_stocks: replace - and . with _)
ticker_table_names = [
    row.ticker.replace("-", "_").replace(".", "_")
    for row in spark.table("stocks.reference.tickers").select("ticker").collect()
]

In [ ]:
for tbl_name in ticker_table_names:
    df = spark.table(f"{staging_schema}.{tbl_name}")
    df = df.withColumn("company", F.lit(tbl_name))

    if not spark.catalog.tableExists("stocks.bronze.stocks_all"):
        df.write.mode("overwrite").format("delta").saveAsTable("stocks.bronze.stocks_all")

    DeltaTable.forName(spark, "stocks.bronze.stocks_all").alias("t").merge(
        df.alias("s"),
        "s.Date = t.Date AND s.company = t.company"
    ).whenNotMatchedInsertAll().execute()